## Init

In [1]:
import requests
from dotenv import dotenv_values
from pathlib import Path


In [2]:
ENV_FILE = dotenv_values(Path('secrets/.env'))

# TOKEN = ENV_FILE.get('TOKEN')
TOKEN = ENV_FILE.get('TOKEN_TEST')
HEADERS = {
    'Authorization': f'Bearer {TOKEN}',
    'Accept-Encoding': 'gzip',
}
LIMIT=100

# В Оркестраторе реализовать:
# В Prefect есть встроенное хранилище переменных. Создать переменную LAST_SUCCESSFUL_SYNC.
# Скрипт читает переменную.
# Если она не задана (или равна None), Prefect понимает, что это «холодный старт», и забирает всё.
# В конце успешного запуска скрипт обновляет эту переменную текущим временем.
START_TIMESTAMP = '2010-04-05 00:00:00'
STOP_TIMESTAMP  = '2027-04-07 00:00:00'

## Формирование RAW Layer

### Get raw data from all sources

In [3]:
params_store = {
    "expand": "zones,slots.zone",
    "filter": f"updated>={START_TIMESTAMP};updated<{STOP_TIMESTAMP}",
    "limit": LIMIT}
params_uom = {
    # "filter": f"updated>={START_TIMESTAMP};updated<{STOP_TIMESTAMP}",  при реальных данных сключить даты
    "limit": LIMIT}
params_product = {
    "expand": "uom,attributes.value",
    "filter": f"updated>={START_TIMESTAMP};updated<{STOP_TIMESTAMP}",
    "limit": LIMIT}
params_variant = {
    "expand": "product",
    "filter": f"updated>={START_TIMESTAMP};updated<{STOP_TIMESTAMP}",
    "limit": LIMIT}
params_agent = {
    "filter": f"updated>={START_TIMESTAMP};updated<{STOP_TIMESTAMP}",
    "limit": LIMIT}
params_in_out = {
    "expand": "positions.slot,positions.assortment,agent",
    "filter": f"updated>={START_TIMESTAMP};updated<{STOP_TIMESTAMP};applicable=true",
    "limit": LIMIT}
params_move = {
    "expand": "positions.targetSlot,positions.sourceSlot,positions.assortment",
    "filter": f"updated>={START_TIMESTAMP};updated<{STOP_TIMESTAMP};applicable=true",
    "limit": LIMIT}


store_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/store', 
    headers=HEADERS, 
    params=params_store).json()

uom_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/uom', 
    headers=HEADERS, 
    params=params_uom).json()

product_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/product', 
    headers=HEADERS, 
    params=params_product).json()
variant_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/variant', 
    headers=HEADERS, 
    params=params_variant).json()

agent_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/counterparty', 
    headers=HEADERS, 
    params=params_agent).json()

demand_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/demand', 
    headers=HEADERS, 
    params=params_in_out).json()
supply_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/supply', 
    headers=HEADERS, 
    params=params_in_out).json()
loss_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/loss', 
    headers=HEADERS, 
    params=params_in_out).json()
enter_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/enter', 
    headers=HEADERS, 
    params=params_in_out).json()

move_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/move', 
    headers=HEADERS, 
    params=params_move).json()


### save .json file to 'temp' for Data Discovery and visual maping

In [4]:
import json

data_to_save = {
    'demand_raw.json': demand_raw,
    'variant_raw.json': variant_raw,
    'product_raw.json': product_raw,
    'store_raw.json': store_raw,
    'enter_raw.json': enter_raw,
    'loss_raw.json': loss_raw,
    'move_raw.json': move_raw,
    'agent_raw.json': agent_raw,
    'uom_raw.json': uom_raw,
}

for filename, content in data_to_save.items():
    file_path = Path('temp/raw_json') / filename
    
    file_path.write_text(
        json.dumps(content, ensure_ascii=False, indent=4), 
        encoding='utf-8'
    )